# **Colabユーザーへの注意**

# **このファイルに直接書き込まないでください—作業が消えることがあります！**

# **必ず作業前にコピーを作成してください。**

コピーの作り方

1. 左上の「File」をクリック  
> *「File」や「Runtime」などのメニューが見えないときは、右上の“v”マークを押して表示してください。*

2. 「Save a copy in Drive」を選ぶ  

3. コピーしたファイル名を「YOURNAMEs_FileName.ipynb」に変更する  
> 例：名前がOliviaなら → Olivias_FileName.ipynb  


---

* チェックマーク（✅）は保存されません。Chromeのリロードボタンでページを更新すると消えます。<br>  
途中で止めるときは、テキストセルを追加して「SO FAR DONE」など書いておいてください。

---

* Colabでは**30分〜90分ごとに以前の出力結果がリセットされます**。<br>  
そのため、`~~ is not defined`のようなエラーが**すごくよく起こります**。

🔁 `~~ is not defined`エラーが出たらどうする？

1. まず変数名のスペルを確認してください。<br>  
2. スペルが正しいのにまだエラーが出るなら、**そのセルをクリックして選択**してください。<br>  
3. 左上の「Runtime」→「Run before」をクリック。<br>  
→ これで**それまでのすべてのセルが再実行されます**。  
4. 再度、そのセルを実行してください。

もしこれでもエラーが直らなければ、<br>  
前のセルのTODOの答えに基本的なミスがあるかもしれません。<br>  
正しいかどうか確認してください。<br>  
またはChatGPTや他のコーディングアシスタントに助けを求めましょう。

# **Preparation**

このセクションでは前のChapterの内容を読み込むだけです。<br>
コードを実行するだけでOK。読まなくても大丈夫です。<br>
気軽に先へ進んでください。<br>

In [ ]:
# ファイルをダウンロードしてください
!wget https://raw.githubusercontent.com/HayatoHongo/Everyones_nanoGPT/main/input.txt -O input.txt
# utf-8でダウンロードしたinput.textファイルを読み込む。
with open("input.txt", 'r', encoding = 'utf-8') as f:
    text = f.read()

# テンソルを見やすく表示する関数（任意）
import torch
import torch.nn as nn
import torch.nn.functional as F

def print_formatted_tensor(*args, width=6, decimals=2):
    """

    A function that neatly formats and displays a PyTorch Tensor, and also prints its size.

    Example usage:
        print_formatted_tensor("名前", tensor)
        print_formatted_tensor(tensor)

    Args:
        *args: If given 1 argument, it is treated as a tensor.
               If given 2 arguments, the first is treated as the name, the second as the tensor.
        width (int): Display width for each number (default: 6)
        decimals (int): Number of decimal places to show (default: 2)
    """


    # 引数からテンソルと名前を判定する
    if not args:
        raise ValueError("At least one argument is required.")
    if isinstance(args[0], str):
        if len(args) < 2:
            raise ValueError("Tensor is not specified.")
        name, tensor = args[0], args[1]
    else:
        name, tensor = None, args[0]

    # Tensorをリストに変換する
    tensor_list = tensor.detach().cpu().tolist()

    def format_list(lst, indent):
        """再帰的ネストリストの整形と文字列返却"""
        # 内容がリストなら再度返す
        if isinstance(lst, list) and lst and isinstance(lst[0], list):
            inner = ",\n".join(" " * indent + format_list(sub, indent + 2) for sub in lst)
            return "[\n" + inner + "\n" + " " * (indent - 2) + "]"
        # 番号付きリスト用
        return "[" + ", ".join(f"{v:{width}.{decimals}f}" for v in lst) + "]"

    # フォーマット済み文字列（最外枠の中括弧は除く）
    formatted = format_list(tensor_list, indent=9)
    inner_formatted = formatted[1:-1].strip()

    # 結果出力
    if name:
        print(name)
    print(f"Tensor Size: {list(tensor.size())}")
    print("tensor([")
    print(" " * 9 + inner_formatted)
    print(" " * 7 + "])")

--2025-10-31 11:44:04--  https://raw.githubusercontent.com/HayatoHongo/Everyones_nanoGPT/main/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.008s  

2025-10-31 11:44:04 (128 MB/s) - ‘input.txt’ saved [1115394/1115394]



# **Chapter 14: Tokens per second(T4 GPU)**



### **Section 1: `tokens_per_second`の測定**

Google Colab では無料のT4 GPUが提供されています。

前のChapterではランタイムがCPUだったので、今回はT4 GPUに設定してください。


念のため確認しましょう。

In [ ]:
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'  # 使用デバイス（GPUまたはCPU）
print(device_type)

cuda


**`Check Point`**  
<label><input type="checkbox">ランタイムがcuda(GPU)に設定されていることを確認した<br></label>  

ここは EveryonesAI サーキット。さあ、CPUとGPUのレースが始まりました。

GPU選手が今まさにスタートを切ろうとしています。

今回も、1秒間に何トークン処理できるかという、`tokens_per_second` を算出します。

Trainerクラスは前のChapterと完全に同一です。

tokens_per_second と統合済みなので、すぐに使えます。

実行するだけで構いません。

In [ ]:
############ NEW ############
import time
############ NEW ############

class Trainer:
    def __init__(self, model, optimizer, data_loader, config):
        self.model = model
        self.optimizer = optimizer
        self.data_loader = data_loader
        self.config = config

    def train_step(self):
        # トレーニング用バッチを取得。
        input_batch, target_batch = self.data_loader.get_batch('train')
        self.optimizer.zero_grad()

        # モデルの順伝播と損失計算
        logits, loss = self.model(input_batch, target_batch)
        loss.backward()  # 誤差逆伝播
        self.optimizer.step()  # パラメータ更新

        return loss.item() # 損失の値を返す

    def evaluate(self):
        self.model.eval()  # 評価モードに切り替え
        losses = {"train": [], "val": []} # 学習・検証データ両方の損失を計算
        with torch.no_grad():
            for split in ['train', 'val']:
                for _ in range(self.config.evaluation_loops):
                    input_batch, target_batch = self.data_loader.get_batch(split)
                    _, loss = self.model(input_batch, target_batch)
                    losses[split].append(loss.item())
        self.model.train()  # 再び学習モードへ戻す

        # 各データセット（train, val）での損失の平均を計算して返す
        return {split: sum(values) / len(values) for split, values in losses.items()}

    def train(self):
        # configで指定された回数だけtrain_stepを実行する。
        for step in range(self.config.total_training_steps):

            # 100回ごと、または最終ステップのみ評価する。
            if step % self.config.evaluation_frequency == 0 or step == self.config.total_training_steps - 1:
                ########## NEW ##########
                # step==0 は last_eval_end_time 未定義のため除外。最終ステップは途中計測になる可能性があるため除外。
                if step == 0 or step == self.config.total_training_steps - 1:
                  tokens_per_second = None
                else:
                  current_eval_start_time = time.time()
                  evaluation_interval = current_eval_start_time - last_eval_end_time
                  tokens_per_evaluation_interval = self.config.batch_size * self.config.input_sequence_length * self.config.evaluation_frequency
                  tokens_per_second = tokens_per_evaluation_interval / evaluation_interval
                ########## NEW ##########

                eval_loss = self.evaluate()
                print(f"Step {step}: Train Loss {eval_loss['train']:.4f}, Validation Loss {eval_loss['val']:.4f}")

                ########## NEW ##########
                print(f"Tokens per second {tokens_per_second}")
                # この評価が終わった時間を記録する。次の評価開始時との時間差が`evaluation_interval`となる。
                last_eval_end_time = time.time()
                ########## NEW ##########

            # 1回の学習ステップ（毎回行う主な処理）
            train_loss = self.train_step()

`Dataloader`クラスとモデルクラスを定義します。Chpater12と完全に同じです。

In [ ]:
class DataLoader:
    def __init__(self, text, config):
        self.config = config  # 設定オブジェクト
        chars = sorted(list(set(text)))  # ユニークな文字をソートする
        self.ctoi = {char: index for index, char in enumerate(chars)}
        self.itoc = {index: char for index, char in enumerate(chars)}
        self.vocab_size = len(chars)

        # エンコードしてテンソルに変換する。
        # `__init__`外のメソッドや引数を呼ぶには`self.`が必要です。
        self.data = torch.tensor(self.encode(text), dtype=torch.long)

        # 訓練用と検証用に分割する。
        # 引数が指定されなくてもself.dataが使われます。
        self.train_data, self.val_data = self.split_data()

    def encode(self, text):
        # 文字列をインデックス列に変換します。self.で他のメソッドや引数を呼び出します。
        return [self.ctoi[c] for c in text]

    def decode(self, indices):
        return ''.join([self.itoc[i] for i in indices])

    def split_data(self):
        split_index = int(0.9 * len(self.data))  # データの90%を訓練用に分割するポイント。
        return self.data[:split_index], self.data[split_index:]

    def get_batch(self, split):
        data = self.train_data if split == 'train' else self.val_data
        start_indices = torch.randint(len(data) - self.config.input_sequence_length, (self.config.batch_size,)) # 抽出開始インデックスを生成する

        input_sequences = torch.stack([
            data[start_index:start_index + self.config.input_sequence_length]
            for start_index in start_indices
        ])
        target_sequences = torch.stack([
            data[start_index + 1:start_index + self.config.input_sequence_length + 1]
            for start_index in start_indices
        ])
        return input_sequences.to(self.config.device_type), target_sequences.to(self.config.device_type)

In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # 語彙数x埋め込み次元の埋め込みテーブルを定義する
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dim)

    def embed(self, input_indices):
        # 入力インデックスに対応する埋め込みベクトルを取得する
        return self.token_embedding_table.forward(input_indices)

class PositionEmbedding(nn.Module):
    def __init__(self, input_sequence_length = 8, embedding_dim = 8):
        super().__init__()
        # 位置埋め込み層
        self.position_embedding_layer = nn.Embedding(input_sequence_length, embedding_dim)

    def forward(self, input_indices):
        # 入力テンソル input_indices の形状：[バッチサイズ、シーケンス長]。
        sequence_length = input_indices.shape[1]

        # シーケンス長に応じた位置インデックスを作成する（例：[0, 1, 2, ..., sequence_length-1]）
        position_indices = torch.arange(sequence_length, device=input_indices.device)

        # 位置インデックスの埋め込みベクトルを取得する
        position_embeddings = self.position_embedding_layer.forward(position_indices)

        return position_embeddings

class EmbeddingModule(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        # 各トークンの埋め込み層
        self.token_embedding_layer = TokenEmbedding(vocab_size = vocab_size, embedding_dim = config.embedding_dim)  # 単語埋め込み層
        self.position_embedding_layer = PositionEmbedding(input_sequence_length = config.input_sequence_length, embedding_dim = config.embedding_dim)  # 位置情報を埋め込む

    def forward(self, input_indices):
        # トークン埋め込みを取得
        token_embeddings = self.token_embedding_layer.embed(input_indices)

        # 位置埋め込みを取得する
        position_embeddings = self.position_embedding_layer.forward(input_indices)

        # トークン埋め込みと位置埋め込みを追加する
        embeddings = position_embeddings + token_embeddings
        return embeddings

class AttentionHead(nn.Module):
    def __init__(self, head_size, config):
        super().__init__()
        self.key_fc= nn.Linear(config.embedding_dim, head_size, bias=False)
        self.query_fc = nn.Linear(config.embedding_dim, head_size, bias=False)
        self.value_fc = nn.Linear(config.embedding_dim, head_size, bias=False)

        # ドロップアウト
        self.dropout = nn.Dropout(config.dropout_rate)
        self.head_size = head_size

    def forward(self, input_tensor):
        B, T, C = input_tensor.shape  # バッチ、トークン長、埋め込みチャネル

        Key = self.key_fc.forward(input_tensor)     # (B, T, head_size)
        Query = self.query_fc.forward(input_tensor)   # (B, T, head_size)
        Value = self.value_fc.forward(input_tensor)   # (B, T, head_size)

        # Attentionスコアを計算中 (QK^T) / sqrt(embedding_dim)
        attention_weights_before_mask = Query @ Key.transpose(-2, -1) * self.head_size**(-0.5)

        # マスク適用済み
        mask = torch.triu(torch.ones(T, T), diagonal=1).to(input_tensor.device)
        masked_attention_weights = attention_weights_before_mask.masked_fill(mask == 1, float('-inf'))

        # ソフトマックス → ドロップアウト → 重み付き和
        attention_weights = F.softmax(masked_attention_weights, dim=-1)
        attention_weights = self.dropout(attention_weights)

        out = attention_weights @ Value  # (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_attention_heads = config.num_attention_heads
        self.embedding_dim = config.embedding_dim
        self.head_size = int(self.embedding_dim / self.num_attention_heads)

        # ModuleListで複数のヘッドを管理する
        self.attention_heads = nn.ModuleList([
            AttentionHead(self.head_size, config)
            for _ in range(self.num_attention_heads)
        ])

        # 各ヘッドの出力を混合する線形層
        self.output_projection = nn.Linear(self.embedding_dim, self.embedding_dim)

        # 出力のドロップアウト
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(self, input_tensor):
        # 各ヘッドの出力を取得する
        # (B, T, head_size)のリスト
        head_outputs_list = [head.forward(input_tensor) for head in self.attention_heads]

        # 全てのヘッドの出力を連結 → (B, T, embedding_dim)
        concatenated = torch.cat(head_outputs_list, dim=-1)

        # 線形変換での出力混合
        projected = self.output_projection.forward(concatenated)

        # 最終出力にドロップアウトを適用する
        output = self.dropout.forward(projected)

        return output

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.embedding_dim, config.hidden_dim),
            nn.ReLU(),
            nn.Linear(config.hidden_dim, config.embedding_dim),
            nn.Dropout(config.dropout_rate),
        )

    def forward(self, input_tensor):
        return self.net(input_tensor)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        # 各LayerNormは独自のbetaとgammaを保持します。
        self.layer_norm1 = nn.LayerNorm(config.embedding_dim)
        self.layer_norm2 = nn.LayerNorm(config.embedding_dim)

        self.multihead_attention = MultiHeadAttention(config=config)
        self.feed_forward = FeedForward(config=config)

    def forward(self, input_tensor):
        # forwardメソッドは省略されています。
        normed_input = self.layer_norm1(input_tensor) # 入力にレイヤーノルムを適用する
        attention_output = self.multihead_attention(normed_input) # マルチヘッドアテンションを適用する
        residual_attention = attention_output + input_tensor # "before! layernorm1"を追加
        normed_attention = self.layer_norm2(residual_attention) # 残差出力に再度LayerNormを適用する
        feedforward_output = self.feed_forward(normed_attention) # フィードフォワードネットワークを適用する
        final_output = feedforward_output + residual_attention # "before" layernorm2 を追加する！

        return final_output

class VocabularyLogits(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        # レイヤー正規化
        self.output_norm = nn.LayerNorm(config.embedding_dim)
        # 語彙数の射影
        self.vocab_projection = nn.Linear(config.embedding_dim, vocab_size)

    def forward(self, transformer_block_output):
        # Transformerブロックの出力にLayer normalizationを適用する。
        normalized_output = self.output_norm.forward(transformer_block_output)  # (B, T, C)

        # 線形層でスコアを語彙数次元に変換する。
        vocab_logits = self.vocab_projection.forward(normalized_output)  # (B, T, V)

        return vocab_logits

class nanoGPT(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.config = config  # 生成時にも使うので保持してください。
        self.embedding = EmbeddingModule(vocab_size, config=config)
        self.blocks = nn.Sequential(*[TransformerBlock(config=config) for _ in range(config.layer_count)])
        self.vocab_projection = VocabularyLogits(vocab_size=vocab_size, config=config)
        self.criterion = nn.CrossEntropyLoss()

    # テキストを生成する
    def generate(self, input_indices, max_new_tokens):
        # 指定したトークン数max_new_tokensのみ生成する
        for _ in range(max_new_tokens):
            input_conditioned = input_indices[:, -self.config.input_sequence_length:] # 入力を切り取る

            # 順伝播は `(likelihood, loss)` を返す—`likelihood` のみを `logits` として保持する。
            logits, _ = self.forward(input_conditioned, target_indices=None)
            last_logits = logits[:, -1, :] # 最後のトークンのロジットを抽出する
            probs = F.softmax(last_logits, dim=-1) # Softmaxで尤度を確率に変換する

            # 次のトークンをサンプリングする
            next_token = torch.multinomial(probs, num_samples=1)

            # 新しいトークンを統合し、input_indicesを更新する。
            input_indices = torch.cat((input_indices, next_token), dim=1)

        # 最終的な`input_indices`を返す。長さは元の`input_indices`＋`max_new_tokens`
        return input_indices

    # 尤度と損失を計算する
    def forward(self, input_indices, target_indices):
        embeddings = self.embedding(input_indices)
        blocks_output = self.blocks(embeddings)
        logits = self.vocab_projection(blocks_output)

        # 推論時はターゲットがないため、lossはNoneです
        # —確率（ロジット）のみ返されます。
        if target_indices is None:
            return logits, None

        batch_size, token_len, vocab_size = logits.shape
        logits = logits.view(batch_size * token_len, vocab_size)
        targets = target_indices.view(batch_size * token_len)
        loss = self.criterion(logits, targets)

        return logits, loss

まずはバッチサイズ1、並列計算なしで測定します。

トレーニングを開始して、`tokens_per_second`を測定しましょう。

最初は、以下の設定で行います。



In [ ]:
# モデル設定を保存する設定クラス
class ModelConfig:
    ########## NEW ##########
    batch_size = 1  # まずはバッチサイズを1に設定する。並列計算はなし。
    input_sequence_length = 512  # データロードにかかる時間の割合を下げるため、長めのシークエンスを一度に取り出す。
    total_training_steps = 150  # あくまでtokens per secondの計測なので、最大ステップ回数は150程度に設定する。
    device_type = 'cuda'  # 使用デバイスはGPUに固定する
    ########## NEW ##########
    evaluation_frequency = 100  # モデル性能評価の頻度
    learning_rate = 0.001  # 学習率
    evaluation_loops = 10  # 評価中の繰り返し回数
    embedding_dim = 64  # 埋め込み層サイズ（特徴ベクトルの次元数）
    hidden_dim = 256
    num_attention_heads = 4  # ノート機構ヘッド番号
    layer_count = 4  # モデルの層数
    dropout_rate = 0.1  # ドロップアウト確率
    random_seed_value = 1337  # 再現性のための乱数シード

In [ ]:
# 設定を読み込みシードを設定する
config = ModelConfig()
torch.manual_seed(config.random_seed_value)  # 再現性確保のため乱数シードを設定

In [ ]:
# データを読み込む
with open("input.txt", 'r', encoding = 'utf-8') as f:
    text_data = f.read()
data_loader = DataLoader(text_data, config)

In [ ]:
# モデルとオプティマイザを初期化する
model = nanoGPT(vocab_size = data_loader.vocab_size, config = config).to(config.device_type)  # 使用するデバイスを指定してください
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

In [ ]:
# モデルのパラメータ数を表示する
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

0.240449 M parameters


In [ ]:
print("===トレーニングが正常に開始されました===")

# モデルを学習する
trainer = Trainer(model, optimizer, data_loader, config)
trainer.train()

===トレーニングが正常に開始されました===
Step 0: Train Loss 4.3347, Validation Loss 4.3314
Tokens per second None
Step 100: Train Loss 2.7626, Validation Loss 2.8232
Tokens per second 13138.304840482022
Step 149: Train Loss 2.7326, Validation Loss 2.6864
Tokens per second None


**`Check Point`**  
<label><input type="checkbox">`tokens_per_second`を確認した<br></label>  

バッチサイズが1だったとしても、GPUはCPUよりも圧倒的に高速です。

GPUは行列演算（＝大量の掛け算＋加算）を一気に並列処理できるからです。

早くもとてつもないスピードを叩き出したGPU。

次は、バッチサイズを1から16に増やしてみて、同様の計算を行います。

In [ ]:
config.batch_size = # TODO: バッチサイズを1から16に増やしてみる

In [ ]:
# モデルとオプティマイザを初期化する
model = nanoGPT(vocab_size = data_loader.vocab_size, config = config).to(config.device_type)  # 使用するデバイスを指定してください
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

In [ ]:
print("===トレーニングが正常に開始されました===")

# モデルを学習する
trainer = Trainer(model, optimizer, data_loader, config)
trainer.train()

===トレーニングが正常に開始されました===
Step 0: Train Loss 4.3887, Validation Loss 4.3837
Tokens per second None
Step 100: Train Loss 2.6432, Validation Loss 2.6323
Tokens per second 139998.86063854798
Step 149: Train Loss 2.5673, Validation Loss 2.5722
Tokens per second None


GPUは並列計算ができるため、処理時間はバッチサイズに比例しません。  

そのため、とんでもない速度を叩き出すことができます。

右上の「RAM・ディスク」横の🔽をクリックし、**「リソースを表示」** を開いてください。


「システムRAM」に加えて、「GPU RAM」というものがあります。これで現在のGPUのメモリ使用率を確認できます。

バッチサイズを増やすと、一度に保持するデータ（テンソル）が増えるため、  
メモリ使用量も増えます。これはCPUと同じです。

バッチサイズをもう少し増やしてみましょう。

In [ ]:
config.batch_size = 128

In [ ]:
# モデルとオプティマイザを初期化する
model = nanoGPT(vocab_size = data_loader.vocab_size, config = config).to(config.device_type)  # 使用するデバイスを指定してください
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

In [ ]:
print("===トレーニングが正常に開始されました===")

# モデルを学習する
trainer = Trainer(model, optimizer, data_loader, config)
trainer.train()

===トレーニングが正常に開始されました===
Step 0: Train Loss 4.2828, Validation Loss 4.2960
Tokens per second None
Step 100: Train Loss 2.6161, Validation Loss 2.6213
Tokens per second 169285.81218587654
Step 149: Train Loss 2.5358, Validation Loss 2.5403
Tokens per second None


あれ、、？😳 なんかあんま速くなってなくね？

はい、そうです。

GPUのメモリの使用率を確認してください。

大きく使用率が上がっています。

先ほどのバッチサイズ1から16に増やした時のように、メモリ使用率がすごく低い状態から使用率を高めると劇的に`tokens per minute`が増えますが、ある程度の使用率からはあまり増えません。

理由は割愛しますが、そういう性質があります。

つまり、限界までバッチサイズを大きくする調整のメリットは大きくありません。

**Section 1: tokens_per_secondの測定** <label><input type="checkbox"> Mark as Done</label>

### **Section 2: Out Of Memory Error**

メモリを使い切りたいと思います。

バッチサイズを極端に増やすと、どうなるでしょう？

In [ ]:
config.batch_size = 1024

In [ ]:
# モデルとオプティマイザを初期化する
model = nanoGPT(vocab_size = data_loader.vocab_size, config = config).to(config.device_type)  # 使用するデバイスを指定してください
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

In [ ]:
print("===トレーニングが正常に開始されました===")

# モデルを学習する
trainer = Trainer(model, optimizer, data_loader, config)
trainer.train()

===トレーニングが正常に開始されました===
Step 0: Train Loss 4.3537, Validation Loss 4.3461
Tokens per second None


OutOfMemoryError: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 748.12 MiB is free. Process 51884 has 14.01 GiB memory in use. Of the allocated memory 12.96 GiB is allocated by PyTorch, and 938.57 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

GPUのメモリを使い切ると、OOM（Out Of Memory）エラーが出ます。


CPUは大きすぎるバッチを自動で小分けに処理できますが、  
GPUは **すべてを一度に載せようとする** ため、メモリが足りないと OOM が発生します。

---

### ⚙️ 対策と目安

| 項目 | 説明 |
|------|------|
| **エラー名** | `OutOfMemoryError`（略して OOM） |
| **原因** | バッチサイズやシーケンス長が大きすぎる |
| **対策** | バッチサイズを減らす／入力を短くする |
| **目安** | GPUメモリ使用率 60〜90% が理想 |

---

### 🔄 OOM発生後の対処

GPUが止まるだけで、**システムRAMは無事**です。  
Colab全体は落ちません。  
再起動するには：

> メニュー → 「ランタイム」 → 「セッションを再起動」

でGPUメモリを解放できます。



#### Colab のGPUの種類

| GPUモデル      | メモリ容量  | プラン／条件            | 目安料金（USD／時間）          |
| ----------- | ------ | ----------------- | --------------------- |
| NVIDIA T4   | 約15 GB | 無料プラン（最大12時間）     | 無料                    |
| NVIDIA L4   | 約24 GB | 有料プラン（アメリカの学生向けに無料プランあり） | 約 **0.18 USD／時間**（目安） |
| NVIDIA A100 | 約40 GB | 有料プラン（アメリカの学生向けに無料プランあり） | 約 **0.76 USD／時間**（目安） |

※ 注：実際の請求レートや「どのGPUが割り当てられるか」はアカウント・時間帯・在庫により異なります。

#### 🙋 課金をするべきか？！

結論：課金しましょう。

まず、A100 40GB が、圧倒的に安いです。

あなたの学費を考えてみてください。ここでケチるのは賢くありません。

A100はT4よりも約3~10倍速いので、自分の時給が 1USD以上 だと思う方は課金しましょう。

また、カリキュラム全体でも、一度の課金(10USD) で十分です。

※注: クレジットは90日の有効期限があるので、再購入を避けるなら3ヶ月以内に終わらせてください！


- 課金プランについて

定額課金制の Colab Pro は **必要ありません。**

従量課金制の[Pay as you Go](https://colab.research.google.com/signup?utm_source=resource_tab&utm_medium=link&utm_campaign=payg_learn_more)を選択しましょう。

---

#### 💸 でもやっぱり課金したくない🥺

安心してください。
このカリキュラムは **無料のT4 GPUだけで十分！**

課金しなくても、Chapter13でお見せしたような昔話生成のLLMがフルスクラッチで作れます✨

---

#### ⚠️ Colab無料枠の限界について

ColabのT4 GPUには**非公開の上限**があります。

2週間で10章くらい進めると、GPUが一時的に使えなくなることも😳

でも大丈夫👇


#### ⚙️ Colab無料枠が切れたときの対策

| 対策                                                     | 内容                                 | デメリット               | 推奨度        |
| :----------------------------------------------------- | :--------------------------------- | :------------------ | :--------- |
| 💰 **10USD だけ課金する（[Pay as you go](https://colab.research.google.com/signup?utm_source=resource_tab&utm_medium=link&utm_campaign=payg_learn_more)）**                                 | 約10 USDでT4 GPUを約50時間以上利用可。全カリキュラムも余裕。 | 少しお金がかかる💸          | ⭐⭐⭐⭐ |
| 🆕 **別のGoogleアカウントを使う**                                | 無料枠を再利用できる。簡単で確実。                  | Driveが分散して管理しづらい。   | ⭐⭐⭐        |
| 💻 **[Kaggle Notebooks](https://www.kaggle.com/code)** | 無料GPUあり（要電話認証）。                    | Google Drive連携が超面倒 | ⭐     |


#### 🧯 タブを閉じない、セッションを切る！

**パソコンを閉じてもGPUは動き続けます⚡**

「よしできた！パソコン💻⤵️」<br>
「よしできた！Colabを閉じて、YouTube見よう▶️」

↑ 一番やっちゃダメ！セッションを必ず切る！

パソコンを閉じても、Colabのタブを閉じても、セッションは走り続ける！

貴重な無料枠をムダに消費してしまうので注意！

終わったら、右上の「🔽」→「ランタイムの接続を解除」をクリック！

---

### ✅ まとめ

| やること              | 理由     |
| :---------------- | :----- |
| T4 GPUでOK         | 無料で十分  |
| 制限かかったら別アカウント| 続行可能   |
| 終わったら接続解除         | GPU節約！ |

これであなたの**無料GPUライフ**は完璧です💪🚀

---

### ⚠️ A100に課金する素晴らしい方々へ

`tokens_per_minute`の測定をA100で行っても、T4 GPUとの大きな差は出ません。<br>
A100のT4に対する優位性は行列演算ですが、今回のConfig設定ではモデルサイズが小さすぎて、<br>
行列のサイズも小さいがために、優位性を発揮できないからです。<br>
モデルサイズが十分に大きくなると、T4の3~10倍くらいの高速性能が出ます。<br>


**Section 2: Out Of Memory Error** <label><input type="checkbox"> Mark as Done</label>

**`Chapter 14: Tokens per second(T4 GPU)`** <label><input type="checkbox"> Mark as Done</label>